In [1]:
import pandas as pd
import re
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# ==================== МЕТАДАННЫЕ (с широтой и долготой) ====================
def get_station_metadata(station_id: str):
    url = f"https://www.pogodaiklimat.ru/station/{station_id}.htm"
    try:
        resp = requests.get(url, timeout=15)
        soup = BeautifulSoup(resp.text, "html.parser")
        text = soup.get_text(separator="\n", strip=True)

        meta = {
            "station_id": station_id,
            "country": "Турция",
            "region": "Турция",
            "station_name": None,
            "lat": None,
            "lon": None,
            "elevation": None
        }

        # Название станции (из заголовка)
        h1 = soup.find("h1")
        if h1:
            m = re.search(r"Метеорологическая станция (.+?)(?: - |/|$)", h1.get_text())
            if m:
                meta["station_name"] = m.group(1).strip()

        # Широта и долгота — улучшенный поиск (учитывает сильный текст и пробелы)
        lat_match = re.search(r"Широта:.*?(\d+\.\d+)", text, re.IGNORECASE | re.DOTALL)
        lon_match = re.search(r"Долгота:.*?(\d+\.\d+)", text, re.IGNORECASE | re.DOTALL)
        
        if lat_match:
            meta["lat"] = float(lat_match.group(1))
        if lon_match:
            meta["lon"] = float(lon_match.group(1))

        # Высота
        elev_match = re.search(r"Высота над уровнем моря:.*?(\d+)", text, re.IGNORECASE | re.DOTALL)
        if elev_match:
            meta["elevation"] = int(elev_match.group(1))

        return meta

    except Exception as e:
        print(f"Ошибка при получении метаданных для станции {station_id}: {e}")
        return {
            "station_id": station_id,
            "country": "Турция",
            "region": "Турция",
            "station_name": None,
            "lat": None,
            "lon": None,
            "elevation": None
        }

# ==================== ПАРСИНГ ТАБЛИЦ ====================
def parse_climate_table(station_id: str, is_precip: bool = False):
    suffix = "_2" if is_precip else ""
    url = f"https://www.pogodaiklimat.ru/history/{station_id}{suffix}.htm"
    prefix = "Os" if is_precip else "Tem"

    try:
        tables = pd.read_html(url, flavor='html5lib')

        # Годы — всегда первая таблица
        years = tables[0].copy()
        years.columns = ["year"]
        years["year"] = pd.to_numeric(years["year"], errors="coerce")

        # Месячные данные — вторая таблица
        data = tables[1].iloc[:, :12].copy()   # только 12 месяцев
        month_names = ["янв","фев","мар","апр","май","июн","июл","авг","сен","окт","ноя","дек"]
        data.columns = month_names

        data = data.apply(pd.to_numeric, errors="coerce")
        data.replace([999.9, -999.9, 9999], pd.NA, inplace=True)

        # Соединяем
        df = pd.concat([years.reset_index(drop=True), data.reset_index(drop=True)], axis=1)

        # Переименовываем в Tem1..Tem12 или Os1..Os12
        rename_dict = {m: f"{prefix}{i+1}" for i, m in enumerate(month_names)}
        df.rename(columns=rename_dict, inplace=True)

        return df

    except Exception as e:
        print(f"Ошибка {'осадков' if is_precip else 'температуры'} {station_id}: {e}")
        cols = ["year"] + [f"{prefix}{i}" for i in range(1,13)]
        return pd.DataFrame(columns=cols)


# ==================== ЗАПУСК ====================
station_ids = ["17150"]
               #, "17234", "17237", ]   # ← добавьте сюда все ваши ID

all_stations = []

for sid in tqdm(station_ids):
    print(f"Обрабатываем {sid}...")
    meta = get_station_metadata(sid)

    temp_df = parse_climate_table(sid, is_precip=False)
    precip_df = parse_climate_table(sid, is_precip=True)

    # Объединяем
    df_station = pd.merge(temp_df, precip_df, on="year", how="outer")

    # Добавляем метаданные
    for k, v in meta.items():
        df_station[k] = v

    # Правильный порядок столбцов
    cols_order = ["station_id", "country", "region", "station_name", 
                  "lat", "lon", "elevation", "year"] + \
                 [f"Tem{i}" for i in range(1,13)] + \
                 [f"Os{i}" for i in range(1,13)]

    df_station = df_station[[c for c in cols_order if c in df_station.columns]]
    all_stations.append(df_station)

final_df = pd.concat(all_stations, ignore_index=True)
final_df = final_df.sort_values(["station_id", "year"]).reset_index(drop=True)

final_df.to_csv("climate_wide_all_stations_FINAL.csv", index=False, encoding="utf-8-sig")

print("\n✅ Готово!")
print("Столбцы:", final_df.columns.tolist())
print(final_df.head(5))

  0%|          | 0/1 [00:00<?, ?it/s]

Обрабатываем 17150...


100%|██████████| 1/1 [00:04<00:00,  4.07s/it]


✅ Готово!
Столбцы: ['station_id', 'country', 'region', 'station_name', 'lat', 'lon', 'elevation', 'year', 'Tem1', 'Tem2', 'Tem3', 'Tem4', 'Tem5', 'Tem6', 'Tem7', 'Tem8', 'Tem9', 'Tem10', 'Tem11', 'Tem12', 'Os1', 'Os2', 'Os3', 'Os4', 'Os5', 'Os6', 'Os7', 'Os8', 'Os9', 'Os10', 'Os11', 'Os12']
  station_id country  region station_name      lat      lon  elevation  \
0      17150  Турция  Турция         None  39.6167  27.9167        101   
1      17150  Турция  Турция         None  39.6167  27.9167        101   
2      17150  Турция  Турция         None  39.6167  27.9167        101   
3      17150  Турция  Турция         None  39.6167  27.9167        101   
4      17150  Турция  Турция         None  39.6167  27.9167        101   

     year  Tem1  Tem2  ...    Os3    Os4   Os5    Os6   Os7   Os8   Os9  Os10  \
0  1937.0   NaN   NaN  ...   13.0   59.0  15.0    0.8  16.0  34.0  10.0  91.0   
1  1938.0   NaN   NaN  ...   38.0   45.0  68.0    5.0   7.0   0.0  43.0  59.0   
2  1939.0   NaN   N